**O3 模型・重點小結**

1. **核心問題**

   * 你要分析一批文本，想先用 **GAP**（Generalized Association Plots）做可視化分群，再用 **LDA**（Latent Dirichlet Allocation）或其監督式變體（如 Labeled LDA）來取得每篇文本的主題分佈，判斷「偏向哪個主題」。

2. **兩種流程比較**

   | 流程 | 先做 GAP → 再做 LDA         | 先指定 topic→token → GAP 分群 → LDA 驗證 |
   | -- | ----------------------- | --------------------------------- |
   | 用途 | 讓可視化主動揭示隱含群，再以 LDA 量化主題 | 用專家/先驗定義 token 群，再檢驗實際分布          |
   | 優點 | 完全資料驅動，能發現意外結構          | 可驗證人為標籤、易對照 domain 知識             |
   | 風險 | GAP 聚類不一定對應可解釋主題        | 預先 token list 可能偏誤，限制 LDA 彈性      |

3. **主題歸屬 → 延伸應用**

   * **推薦系統**：把「文件–主題分布」對應成「使用者/商品–潛在偏好」，即可做內容推薦或商品推薦。
   * **顧客分析**：將每位顧客的行為紀錄視為文件，LDA 得到「顧客–主題偏好向量」，GAP 幫你做群組視覺化，便於客群分層、行銷定位。

4. **實作建議**

   1. **訓練**：對已標或未標文本跑 LDA（或 Labeled LDA 如果有多重標籤）。
   2. **推論**：輸入新文本，推斷其 `θ`（topic mixture），即得到「偏向哪個主題」的機率。
   3. **可視化／分群**：用 GAP 依 `θ` 或相似度排序，觀察群落與主題塊狀。
   4. **應用**：

      * 推薦：以使用者對主題的偏好去匹配文件或商品。
      * 分群：以顧客在主題空間的位置做 segmentation。

5. **下一步**

   * 若想快速驗證，可先用現成 LDA 套件（Gensim／sklearn）＋ R 版 GAP。
   * 需要設計推薦或顧客分析流程，再進一步整合「主題 → 相似度 → 排薦」的演算法。


In [67]:
import pandas as pd
import re, string

# 下載並設定停用詞
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
stop_words = stopwords.words('english')

custom_stop = {'said', 'mr', 'also', 'would', 'people', 'new', 'one', 'like', 'could', 'us', 'many', 'get', 'use'}
all_stopwords = set(stop_words).union(custom_stop)

from collections import Counter, defaultdict
from sklearn.feature_extraction.text import CountVectorizer
data = pd.read_csv(r'C:\Users\USER\Documents\GitHub\psychic-spoon\BERTopic\bbc-news-data.csv', sep='\t')

def preprocess(text):
    text = text.lower()  # 轉成小寫
    text = re.sub(r'\d+', '', text)  # 移除數字
    text = re.sub(r'[^\w\s]', '', text)  # 移除標點符號
    text = re.sub(r'\s+', ' ', text).strip()  # 移除多餘空白
    text = ' '.join([word for word in text.split() if word not in all_stopwords])  # 移除停用詞
    return text

data['cleaned_content'] = data['content'].apply(preprocess)
data['Tokens'] = data['cleaned_content'].apply(lambda x: x.split())

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\USER\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


---

### 自定義seed

In [ ]:
seed = {
    'business':['market', 'share', 'profit', 'loss', 'company', 'merger', 'revenue', 'investor', 'bank', 'finance', 'economy'],
    'entertaiment':['film', 'movie', 'music', 'album', 'actor', 'actress', 'show', 'theatre', 'festival', 'award', 'star'],
    'politics':['government', 'minister', 'election', 'parliament', 'policy', 'vote', 'party', 'labour', 'conservative', 'law'],
    'sport':['match', 'win', 'defeat', 'league', "cup", 'coach', 'player', 'score', 'goal', 'season'],
    'tech':['software', 'internet', 'computer', 'mobile', 'digital', 'iphone', 'microsoft', 'google', 'technology', 'device']
}

selected_tokens = set()
for words in seed.values():
    selected_tokens.update(words)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=list(selected_tokens))
X = vectorizer.fit_transform(data['cleaned_content'])

tf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tf_df.to_csv("filtered_doc_token_matrix3.csv", index=False)



---

### 算文章的tfidf來設定seed

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

for label in data['category'].unique():
    docs = data[data['category'] == label]['cleaned_content']
    vec = TfidfVectorizer(max_df=0.15, min_df=20, stop_words='english')
    tfidf = vec.fit_transform(docs)
    top_words = sorted(zip(vec.get_feature_names_out(), 
                           tfidf.sum(axis=0).A1),
                           key=lambda x: -x[1])[:30]
    print(f'Top words for {label}: {[w for w, _ in top_words]}')

In [ ]:
seed = {
    'business':['profits', 'yukos', 'dollar', 'euros', 'china', 'rates', 'countries', 'spending', 'foreign', 'offer', 'million', 'share', 'jobs', 'tax', 'production', 'bid', 'car', 'report', 'reported', 'fell', 'debt', 'public', 'europe', 'exports', 'budget', 'profit', 'deutsche', 'london', 'consumer', 'banks'],
    'entertaiment':['festival', 'chart', 'rock', 'song', 'office', 'told', 'book', 'role', 'oscar', 'box', 'nominations', 'single', 'musical', 'starring', 'theatre', 'group', 'television', 'sales', 'released', 'drama', 'radio', 'sold', 'ceremony', 'love', 'im', 'industry', 'named', 'career', 'pop', 'christmas'],
    'politics':['lord', 'police', 'eu', 'kennedy', 'committee', 'immigration', 'lords', 'rights', 'services', 'iraq', 'bn', 'education', 'human', 'politics', 'dems', 'spending', 'children', 'men', 'budget', 'cabinet', 'asylum', 'european', 'taxes', 'meeting', 'conference', 'wales', 'ms', 'blunkett', 'scottish', 'schools'],
    'sport':['wales', 'chelsea', 'rugby', 'united', 'williams', 'seed', 'olympic', 'liverpool', 'race', 'australian', 'scotland', 'european', 'arsenal', 'squad', 'goal', 'robinson', 'champions', 'grand', 'ball', 'try', 'round', 'jones', 'italy', 'record', 'athens', 'real', 'madrid', 'decision', 'half', 'ive'],
    'tech':['search', 'apple', 'sites', 'email', 'report', 'gaming', 'radio', 'programs', 'bt', 'control', 'european', 'windows', 'mobiles', 'sony', 'spam', 'legal', 'program', 'google', 'project', 'files', 'virus', 'version', 'messages', 'wireless', 'released', 'gadgets', 'device', 'law', 'images', 'consumer']
}


In [ ]:

selected_tokens = set()
for words in seed.values():
    selected_tokens.update(words)

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(vocabulary=list(selected_tokens))
X = vectorizer.fit_transform(data['cleaned_content'])

tf_df = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
tf_df.to_csv("篩選各topic的tfidf前30.csv", index=False)


---

---

---

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, adjusted_rand_score, classification_report
import pandas as pd
import numpy as np

def run_gap_informed_lda(data, token_column='Tokens', label_column='category', 
                         token_to_topic_file='篩選各topic的tfidf前30的token分配主題.csv',
                         K=5, eta_strength=0.3, iterations=100, passes=10):
    """
    完整執行 GAP-informed LDA 主題建模 + 分類效果評估
    """

    # 1. 建立詞典與語料
    dictionary = Dictionary(data[token_column])
    corpus = [dictionary.doc2bow(tokens) for tokens in data[token_column]]

    # 2. 讀取 token-to-topic mapping
    token_df = pd.read_csv(token_to_topic_file)
    token_to_topic = dict(zip(token_df['token'], token_df['topic']))

    # 3. 建立 η 矩陣
    V = len(dictionary.token2id)
    eta = np.full((K, V), 0.01)
    for word, topic_id in token_to_topic.items():
        if topic_id < K and word in dictionary.token2id:
            word_id = dictionary.token2id[word]
            eta[topic_id][word_id] = eta_strength

    # 4. 建立 LDA 模型
    lda = LdaModel(corpus=corpus, id2word=dictionary, num_topics=K,
                   eta=eta, random_state=42, iterations=iterations, passes=passes)

    # 5. 主題詞顯示
    print("\n每個主題的前 20 詞：")
    for k in range(K):
        top_words = lda.show_topic(k, topn=20)
        print(f"Topic {k+1}: {[w for w, _ in top_words]}")

    # 6. 主題預測
    pred_topic = [
        max(lda.get_document_topics(doc), key=lambda x: x[1])[0]
        for doc in corpus
    ]

    # 7. 對真實分類做 label encoding
    le = LabelEncoder()
    true_topic = le.fit_transform(data[label_column])

    # 8. 評估指標
    acc = accuracy_score(true_topic, pred_topic)
    ari = adjusted_rand_score(true_topic, pred_topic)
    report = classification_report(true_topic, pred_topic, target_names=le.classes_)

    print("\nAccuracy:", round(acc, 4))
    print("Adjusted Rand Index (ARI):", round(ari, 4))
    print("\nClassification Report:\n", report)

    # 9. 回傳必要結果
    data['pred_topic'] = pred_topic
    return lda, dictionary, corpus, eta, data



In [71]:
lda_model, dictionary, corpus, eta_matrix, result_df = run_gap_informed_lda(data)



📌 每個主題的前 10 詞：
Topic 1: ['mobile', 'phone', 'software', 'government', 'phones', 'technology', 'users', 'security', 'system', 'uk', 'services', 'last', 'says', 'year', 'report', 'number', 'using', 'make', 'access', 'programs']
Topic 2: ['year', 'film', 'best', 'years', 'million', 'first', 'sales', 'growth', 'bn', 'market', 'last', 'uk', 'economy', 'figures', 'made', 'director', 'economic', 'awards', 'two', 'time']
Topic 3: ['digital', 'tv', 'broadband', 'technology', 'users', 'search', 'service', 'content', 'services', 'media', 'apple', 'online', 'music', 'market', 'video', 'first', 'net', 'uk', 'mobile', 'want']
Topic 4: ['two', 'last', 'told', 'first', 'win', 'world', 'back', 'england', 'three', 'bn', 'years', 'time', 'game', 'party', 'government', 'added', 'think', 'former', 'made', 'deal']
Topic 5: ['music', 'technology', 'game', 'games', 'data', 'used', 'mobile', 'make', 'time', 'world', 'networks', 'users', 'net', 'firms', 'video', 'way', 'radio', 'email', 'says', 'computer']

 A